In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#  Install all dependencies 

import sys, warnings, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

print('Installing NLP metrics and core dependencies...')
!pip install -q --upgrade pip
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    sacrebleu==2.4.2 \
    rouge-score==0.1.2 \
    evaluate==0.4.2 \
    bert-score==0.3.13 \
    jiwer==3.0.4 \
    nltk==3.9.1 \
    sentence-transformers==3.0.1 \
    rank-bm25==0.2.2 \
    tqdm pandas numpy scipy scikit-learn matplotlib seaborn

print('Installing FAISS GPU...')
# Tries CUDA 12 GPU faiss, falls back to CUDA 11 if needed
!pip install -q faiss-gpu-cu12 || pip install faiss-gpu

print('All packages installed successfully ✓')

In [ ]:
!pip install -q -U bitsandbytes

In [1]:
import torch
import transformers
import accelerate
import peft
import sacrebleu
import faiss

print("=" * 55)
print(f"✓ PyTorch version   : {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"✓ Transformers      : {transformers.__version__}")
print(f"✓ SacreBLEU         : {sacrebleu.__version__}")
print(f"✓ FAISS GPU Devices : {faiss.get_num_gpus()}")
print("=" * 55)
print("All core libraries are healthy and ready to go! ✓")

✓ PyTorch version   : 2.10.0+cu128 (CUDA: True)
✓ Transformers      : 4.44.2
✓ SacreBLEU         : 2.4.2
✓ FAISS GPU Devices : 2
All core libraries are healthy and ready to go! ✓


In [2]:
#  Airtight Global Noise & Warning Suppression Protocol 
# This block MUST run at the top of every subsequent cell.
import sys, warnings, logging, os

#  Bypass protobuf/wandb version clashes BEFORE any NLP import
sys.modules['wandb'] = None

#  Suppress all Python warnings (deprecations, user warnings, etc.)
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '0'

#  Kill HuggingFace widget progress bars (prevents "Error displaying widget")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

#  Silence all relevant loggers to ERROR-only
_NOISY_LOGGERS = [
    'transformers', 'transformers.generation.utils',
    'transformers.tokenization_utils_base',
    'transformers.modeling_utils',
    'sentence_transformers', 'torch',
    'evaluate', 'datasets', 'peft',
    'accelerate', 'bitsandbytes',
    'filelock', 'urllib3', 'requests', 'absl',
]
for _n in _NOISY_LOGGERS:
    logging.getLogger(_n).setLevel(logging.ERROR)

#  Root-level safety net
logging.basicConfig(level=logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)



In [3]:
#  Global Imports, Configuration & Reproducibility
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
for _n in ['transformers','sentence_transformers','torch','evaluate',
           'transformers.generation.utils','datasets','peft','accelerate']:
    logging.getLogger(_n).setLevel(logging.ERROR)

import gc, json, re, math, time, random
from pathlib import Path
from glob import glob
from copy import deepcopy
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch

#  Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#  GPU Configuration
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'[CONFIG] GPUs available: {NUM_GPUS}')
for _i in range(NUM_GPUS):
    _mem = torch.cuda.get_device_properties(_i).total_memory / 1e9
    print(f'  GPU {_i}: {torch.cuda.get_device_name(_i)} | {_mem:.1f} GB VRAM')

# Output Paths 
WORK_DIR  = Path('/kaggle/working')
MODEL_DIR = WORK_DIR / 'fine_tuned_dialect_model'
FIG_DIR   = WORK_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

#  Canonical Column Names 
COL_SRC     = 'source'
COL_TGT     = 'target'
COL_DIALECT = 'dialect_type'

print(' Global configuration complete.')

[CONFIG] GPUs available: 2
  GPU 0: Tesla T4 | 15.6 GB VRAM
  GPU 1: Tesla T4 | 15.6 GB VRAM
 Global configuration complete.


In [4]:
# Resolve & Load Dataset Files 

sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

from glob import glob
from pathlib import Path
import pandas as pd

WORK_DIR = Path('/kaggle/working')

# Explicit Column Mapping for Kothon Dialect CSVs 
COLUMN_MAP = {
    'dialect_text':    'source',        
    'standard_bangla': 'target',        
    'dialect_name':    'dialect_type',  
}

def _resolve_csv(pattern: str, fallback_dialect: str) -> pd.DataFrame:
    matches = glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find '{pattern}' anywhere under /kaggle/input/")
        
    filepath = matches[0]
    print(f'  [FOUND] {filepath}')
    
    df = pd.read_csv(filepath, encoding='utf-8')
    
    # Rename columns to standard names
    df = df.rename(columns=COLUMN_MAP)
    
    # Ensure dialect_type is populated
    if 'dialect_type' not in df.columns or df['dialect_type'].isna().all():
        df['dialect_type'] = fallback_dialect
        
    # Keep only the required columns and clean text
    df = df[['source', 'target', 'dialect_type']].dropna()
    df['source'] = df['source'].astype(str).str.strip()
    df['target'] = df['target'].astype(str).str.strip()
    df['dialect_type'] = df['dialect_type'].astype(str).str.strip().str.lower()
    
    # Filter empty or corrupt rows
    df = df[(df['source'] != '') & (df['source'] != 'nan') &
            (df['target'] != '') & (df['target'] != 'nan')].reset_index(drop=True)
            
    return df

print('Resolving dataset files...')
df_chitt  = _resolve_csv('chittagonian_preprocessed.csv', 'chittagonian')
df_sylhet = _resolve_csv('sylheti_preprocessed.csv', 'sylheti')
df_all    = pd.concat([df_chitt, df_sylhet], ignore_index=True)

print(f'\n[DATA LOADED SUCCESSFULLY ✓]')
print(f'  • Chittagonian: {len(df_chitt):,} pairs')
print(f'  • Sylheti     : {len(df_sylhet):,} pairs')
print(f'  • Total       : {len(df_all):,} pairs\n')

# Display first 5 samples
df_all.head()

Resolving dataset files...
  [FOUND] /kaggle/input/datasets/iamimran7/dialect/chittagonian_preprocessed.csv
  [FOUND] /kaggle/input/datasets/iamimran7/dialect/sylheti_preprocessed.csv

[DATA LOADED SUCCESSFULLY ✓]
  • Chittagonian: 7,871 pairs
  • Sylheti     : 9,169 pairs
  • Total       : 17,040 pairs



,source,target,dialect_type
0,চাটগাঁইয়া ডঅর চাটগাঁ অঞ্চলর আঞ্চলিক ভাষা।,চাটগাঁইয়া বৃহত্তর চট্টগ্রাম অঞ্চলের আঞ্চলিক ভাষা।,chittagonian
1,বাঁআলা ভাষার হোঁয়ারে চাটগাঁইয়া ভাষার বউত শব্দর...,বাংলা ভাষার সঙ্গে চাটগাঁইয়া ভাষার শাব্দিক সম্প...,chittagonian
2,তয় কেউ কেউ এ ভাষারে ওউগগা স্বাধীন ভাষার মজ্জাদ...,তবে কেউ কেউ একে স্বতন্ত্র ভাষার মর্যাদা দিতে উ...,chittagonian
3,চাটগাঁইয়া আদতে মুখর ভাষা।,চাটগাঁইয়া মূলত মৌখিক ভাষা।,chittagonian
4,ইবা নিজর মতো লেকনর কন কুদরত নাই।,এর নিজস্ব লিপি পদ্ধতি নেই।,chittagonian


In [ ]:
import sys, warnings, logging, os
from pathlib import Path
from sklearn.model_selection import train_test_split
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
#  90% temp / 10% test (Stratified by dialect_type)
_temp, test_df = train_test_split(
    df_all, test_size=0.10,
    stratify=df_all['dialect_type'], random_state=42
)
#  80% train / 10% val (11.11% of 90% = 10% of total)
train_df, val_df = train_test_split(
    _temp, test_size=0.1111,
    stratify=_temp['dialect_type'], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
# Save splits to disk
train_df.to_csv(WORK_DIR / 'split_train.csv', index=False)
val_df.to_csv(WORK_DIR   / 'split_val.csv',   index=False)
test_df.to_csv(WORK_DIR  / 'split_test.csv',  index=False)
#  Build training vocabulary (Essential for Difficulty Estimator in RAG)
train_vocab = set([w for s in train_df['source'] for w in s.split()])
print('[SPLIT] Stratified splits created and saved:')
for _nm, _df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    _vc = _df['dialect_type'].value_counts()
    print(f'  • {_nm:5s}: {len(_df):5,} rows | Chittagonian: {_vc.get("chittagonian", 0):4,} | Sylheti: {_vc.get("sylheti", 0):4,}')
print(f'\n[VOCAB] Unique training vocabulary size: {len(train_vocab):,} words')
# Expose flat lists for inference
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'
_test_sources   = test_df[COL_SRC].tolist()
_test_targets   = test_df[COL_TGT].tolist()
_test_dialects  = test_df[COL_DIALECT].tolist()
_train_sources  = train_df[COL_SRC].tolist()
_train_targets  = train_df[COL_TGT].tolist()
_train_dialects = train_df[COL_DIALECT].tolist()
print('[SPLIT] Complete ✓')

[SPLIT] Stratified splits created and saved:
  • Train: 13,632 rows | Chittagonian: 6,297 | Sylheti: 7,335
  • Val  : 1,704 rows | Chittagonian:  787 | Sylheti:  917
  • Test : 1,704 rows | Chittagonian:  787 | Sylheti:  917

[VOCAB] Unique training vocabulary size: 22,886 words
[SPLIT] Complete ✓


In [6]:
# Unified Metric Engine 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','evaluate','bert_score','jiwer','rouge_score','nltk']:
    logging.getLogger(_n).setLevel(logging.ERROR)

import sacrebleu
import evaluate as hf_evaluate
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.meteor_score import meteor_score as _nltk_meteor
from bert_score import score as bert_score_fn
import jiwer
import pandas as pd
from typing import List, Dict

# Load Hugging Face ROUGE Metric
_hf_rouge = hf_evaluate.load("rouge")

BERTSCORE_MODEL = 'bert-base-multilingual-cased'  # Standard supported model for Bengali BERTScore


def compute_all_metrics(
    hypotheses: List[str],
    references: List[str],
    compute_bertscore: bool = True,
    bertscore_device: str = 'cuda:0',
) -> Dict[str, float]:
    """
    Compute all 10 evaluation metrics robustly for Bengali.
    """
    _pairs = [(str(h).strip(), str(r).strip()) for h, r in zip(hypotheses, references) if str(h).strip() and str(r).strip()]
    if not _pairs:
        return {m: 0.0 for m in ['bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4', 
                                 'rouge', 'rouge1', 'rouge2', 'rougeL',
                                 'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1']}
    hyps, refs = zip(*_pairs)
    hyps, refs = list(hyps), list(refs)
    res = {}

    #  SacreBLEU (Overall Standard BLEU)
    try:
        _sc_std = sacrebleu.corpus_bleu(hyps, [refs], smooth_method='exp', tokenize='flores200')
        res['bleu'] = round(_sc_std.score, 4)
    except Exception:
        res['bleu'] = 0.0

    #  Individual N-gram BLEU (1, 2, 3, 4)
    for _ord, _key in [(1, 'bleu1'), (2, 'bleu2'), (3, 'bleu3'), (4, 'bleu4')]:
        try:
            _bleu_obj = sacrebleu.BLEU(max_ngram_order=_ord, tokenize='flores200', smooth_method='exp')
            _sc = _bleu_obj.corpus_score(hyps, [refs])
            res[_key] = round(_sc.score, 4)
        except Exception:
            res[_key] = 0.0

    #  ROUGE 
    try:
        rouge_results = _hf_rouge.compute(
            predictions=hyps, 
            references=refs, 
            tokenizer=lambda x: x.split()  # Splitting Bengali text natively by spaces
        )
        r1 = rouge_results['rouge1'] * 100
        r2 = rouge_results['rouge2'] * 100
        rl = rouge_results['rougeL'] * 100
        
        res['rouge1'] = round(r1, 4)
        res['rouge2'] = round(r2, 4)
        res['rougeL'] = round(rl, 4)
        res['rouge']  = round((r1 + r2 + rl) / 3.0, 4)
    except Exception as e:
        print(f"[ROUGE Warning] {e}")
        res['rouge'], res['rouge1'], res['rouge2'], res['rougeL'] = 0.0, 0.0, 0.0, 0.0

    #  METEOR
    _met = []
    for h, r in zip(hyps, refs):
        try: _met.append(_nltk_meteor([r.split()], h.split()))
        except Exception: _met.append(0.0)
    res['meteor'] = round(float((sum(_met) / len(_met)) * 100), 4)

    #  chrF++
    try:
        res['chrfpp'] = round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 4)
    except Exception:
        res['chrfpp'] = 0.0

    #  WER & WIL
    try:
        _wer_out = jiwer.process_words(refs, hyps)
        res['wer'] = round(_wer_out.wer, 4)
        res['wil'] = round(_wer_out.wil, 4)
    except Exception:
        res['wer'], res['wil'] = 1.0, 1.0

    #  BERTScore
    if compute_bertscore:
        try:
            _, _, _F1 = bert_score_fn(
                hyps, refs, model_type=BERTSCORE_MODEL,
                lang='bn', device=bertscore_device,
                verbose=False, batch_size=32
            )
            res['bertscore_f1'] = round(float(_F1.mean().item() * 100), 4)
        except Exception as e:
            res['bertscore_f1'] = 0.0
    else:
        res['bertscore_f1'] = float('nan')

    return res


def evaluate_by_dialect(
    hypotheses: List[str],
    references: List[str],
    dialect_labels: List[str],
    system_name: str = 'system',
    compute_bertscore: bool = True,
) -> pd.DataFrame:
    """Compute all 10 metrics per dialect + overall."""
    rows = []
    for _d in ['chittagonian', 'sylheti', 'overall']:
        if _d == 'overall':
            _h, _r = hypotheses, references
        else:
            _idx = [i for i, d in enumerate(dialect_labels) if d == _d]
            _h   = [hypotheses[i] for i in _idx]
            _r   = [references[i] for i in _idx]
        _m = compute_all_metrics(_h, _r, compute_bertscore)
        _m.update({'system': system_name, 'dialect': _d, 'n_samples': len(_h)})
        rows.append(_m)
    return pd.DataFrame(rows)


print('Unified Metric Engine ready (BLEU, BLEU-1/2/3/4, ROUGE, ROUGE-1/2/L, METEOR, chrF++, WER, WIL, BERTScore-F1).')

Unified Metric Engine ready (BLEU, BLEU-1/2/3/4, ROUGE, ROUGE-1/2/L, METEOR, chrF++, WER, WIL, BERTScore-F1).


In [7]:
# Stable NLLB-600M Fine-Tuning 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','accelerate','peft']: logging.getLogger(_n).setLevel(logging.ERROR)

import gc, torch, pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

WORK_DIR = Path('/kaggle/working')
FT_MODEL_DIR = WORK_DIR / 'fine_tuned_nllb600m'
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def _flush_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

_flush_gpu()

def _safe_seq2seq(model, tokenizer, texts, forced_bos_token_id=None, max_new_tokens=128, batch_size=32, device=DEVICE):
    """Batched inference without pipeline overhead."""
    model.eval()
    all_preds = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=128
        ).to(device)
        
        with torch.inference_mode():
            gen_kwargs = {
                "max_new_tokens": max_new_tokens,
                "num_beams": 4, 
                "early_stopping": True
            }
            if forced_bos_token_id is not None:
                gen_kwargs["forced_bos_token_id"] = forced_bos_token_id
                
            outputs = model.generate(**inputs, **gen_kwargs)
            
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_preds.extend([txt.strip() for txt in decoded])
        
    return all_preds



# Initialize Tokenizer 
MODEL_NAME = "facebook/nllb-200-distilled-600M"
print(f"[SETUP] Initializing Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.src_lang = "ben_Beng"
tokenizer.tgt_lang = "ben_Beng"

forced_bos_token_id = tokenizer.convert_tokens_to_ids("ben_Beng")

def _preprocess(examples):
    inputs = [str(ex) for ex in examples['source']]
    targets = [str(ex) for ex in examples['target']]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("[SETUP] Mapping parallel datasets...")
hf_train = Dataset.from_pandas(train_df[['source', 'target']]).map(_preprocess, batched=True, remove_columns=['source', 'target'])
hf_val = Dataset.from_pandas(test_df[['source', 'target']]).map(_preprocess, batched=True, remove_columns=['source', 'target'])

# Load Model in FP32 & Apply LoRA 
print("[SETUP] Loading base model in stable float32...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32, 
    low_cpu_mem_usage=True
)

# Ensure gradient checkpointing is completely disabled to avoid DataParallel crashes
base_model.config.use_cache = True

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

# Training Configuration 
training_args = Seq2SeqTrainingArguments(
    output_dir=str(FT_MODEL_DIR),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8, 
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=4, 
    predict_with_generate=False, 
    fp16=False, 
    bf16=False,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
    gradient_checkpointing=False, # EXPLICITLY DISABLED
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train Model 
print("\n" + "="*80)
print("[TRAINING] Starting Stable NLLB-600M Fine-Tuning (4 Epochs)...")
print("="*80)
trainer.train()

# Save Model
print("\n[SAVING] Saving LoRA adapters and tokenizer to disk...")
model.save_pretrained(FT_MODEL_DIR)
tokenizer.save_pretrained(FT_MODEL_DIR)
print(f"  • Model checkpoint successfully saved to: {FT_MODEL_DIR} ✓")

del trainer, base_model
_flush_gpu()

#  Full Inference on Test Set 
print("\n" + "="*80)
print("[EVALUATION] Generating predictions on full test set (1,704 samples)...")
print("="*80)

model = model.to(DEVICE)
_test_sources = test_df['source'].tolist()

# Batched inference
nllb_ft_preds = _safe_seq2seq(
    model, tokenizer, _test_sources, 
    forced_bos_token_id=forced_bos_token_id, 
    max_new_tokens=128, 
    batch_size=32, 
    device=DEVICE
)

_out_df = test_df.copy()
_out_df['pred_nllb_ft'] = nllb_ft_preds
_out_path = WORK_DIR / 'nllb_fine_tuned_predictions.csv'
_out_df.to_csv(_out_path, index=False, encoding='utf-8-sig')

print(f"\n[DONE] Fine-tuned predictions saved to: {_out_path}")
print(_out_df[['source', 'target', 'pred_nllb_ft']].head(3).to_markdown(index=False))

[SETUP] Initializing Tokenizer for facebook/nllb-200-distilled-600M...
[SETUP] Mapping parallel datasets...


Map:   0%|          | 0/13632 [00:00<?, ? examples/s]

Map:   0%|          | 0/1704 [00:00<?, ? examples/s]

[SETUP] Loading base model in stable float32...
trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821

[TRAINING] Starting Stable NLLB-600M Fine-Tuning (4 Epochs)...
{'loss': 1.9972, 'grad_norm': 0.2197122424840927, 'learning_rate': 0.0002823943661971831, 'epoch': 0.2347417840375587}
{'loss': 1.3488, 'grad_norm': 0.2245195508003235, 'learning_rate': 0.00026478873239436614, 'epoch': 0.4694835680751174}
{'loss': 1.2373, 'grad_norm': 0.2867622673511505, 'learning_rate': 0.00024718309859154925, 'epoch': 0.704225352112676}
{'loss': 1.1502, 'grad_norm': 0.2515406906604767, 'learning_rate': 0.0002295774647887324, 'epoch': 0.9389671361502347}
{'eval_loss': 1.051544189453125, 'eval_runtime': 69.4718, 'eval_samples_per_second': 24.528, 'eval_steps_per_second': 3.066, 'epoch': 1.0}
{'loss': 1.1168, 'grad_norm': 0.3591879606246948, 'learning_rate': 0.00021197183098591548, 'epoch': 1.1737089201877935}
{'loss': 1.0598, 'grad_norm': 0.363353967666626, 'learning_rate': 0.0001943

In [8]:
#  Compute All Metrics for Fine-Tuned NLLB-600M 
import sys, warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
from pathlib import Path

WORK_DIR = Path('/kaggle/working')

# Load Fine-Tuned Predictions
pred_file = WORK_DIR / 'nllb_fine_tuned_predictions.csv'

if 'nllb_ft_preds' in globals() and '_out_df' in globals():
    eval_df = _out_df.copy()
    print("[LOAD] Using in-memory fine-tuned predictions (_out_df) ✓")
elif pred_file.exists():
    eval_df = pd.read_csv(pred_file)
    print(f"[LOAD] Loaded predictions from: {pred_file} ✓")
else:
    raise FileNotFoundError(f"Predictions file not found at {pred_file}. Please ensure training/inference has completed.")

# Standardize column names
src_col = 'source' if 'source' in eval_df.columns else eval_df.columns[0]
tgt_col = 'target' if 'target' in eval_df.columns else eval_df.columns[1]
dialect_col = 'dialect_type' if 'dialect_type' in eval_df.columns else 'dialect'
pred_col = 'pred_nllb_ft' if 'pred_nllb_ft' in eval_df.columns else 'prediction'

hypotheses = eval_df[pred_col].astype(str).tolist()
references = eval_df[tgt_col].astype(str).tolist()
dialects   = eval_df[dialect_col].astype(str).tolist()

print(f"[EVAL] Evaluating {len(hypotheses)} samples across dialects using Unified Metric Engine...")

#  Run Comprehensive Metric Evaluation 
_nllb_ft_metrics = evaluate_by_dialect(
    hypotheses=hypotheses,
    references=references,
    dialect_labels=dialects,
    system_name='Fine-Tuned NLLB-600M',
    compute_bertscore=True
)

# Reorder columns standardly
_col_ord = [
    'system', 'dialect', 'n_samples',
    'bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4',
    'rouge', 'rouge1', 'rouge2', 'rougeL',
    'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1'
]
_nllb_ft_metrics = _nllb_ft_metrics[[c for c in _col_ord if c in _nllb_ft_metrics.columns]]

#  Save Files Accordingly
out_csv_path  = WORK_DIR / 'nllb_fine_tuned_metrics.csv'
out_xlsx_path = WORK_DIR / 'nllb_fine_tuned_metrics.xlsx'

_nllb_ft_metrics.to_csv(out_csv_path, index=False, encoding='utf-8-sig')
_nllb_ft_metrics.to_excel(out_xlsx_path, index=False, engine='openpyxl')

print(f"\n[SAVED] CSV saved to   : {out_csv_path} ✓")
print(f"[SAVED] Excel saved to : {out_xlsx_path} ✓")

# Formatted Display 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n" + "="*120)
print("                   [FINE-TUNED NLLB-600M STANDALONE PERFORMANCE BENCHMARK]")
print("="*120)
print(_nllb_ft_metrics.to_string(index=False))

#  Raw CSV Output
print("\n" + "#"*100)
print("--- [SAFETY NET: RAW CSV TEXT OF nllb_fine_tuned_metrics.csv] ---")
print("#"*100)
print(_nllb_ft_metrics.to_csv(index=False))
print("#"*100)
print("--- [END OF RAW CSV TEXT] ---\n")

[LOAD] Using in-memory fine-tuned predictions (_out_df) ✓
[EVAL] Evaluating 1704 samples across dialects using Unified Metric Engine...

[SAVED] CSV saved to   : /kaggle/working/nllb_fine_tuned_metrics.csv ✓
[SAVED] Excel saved to : /kaggle/working/nllb_fine_tuned_metrics.xlsx ✓

                   [FINE-TUNED NLLB-600M STANDALONE PERFORMANCE BENCHMARK]
              system      dialect  n_samples    bleu   bleu1   bleu2   bleu3   bleu4   rouge  rouge1  rouge2  rougeL  meteor  chrfpp    wer    wil  bertscore_f1
Fine-Tuned NLLB-600M chittagonian        787 61.1510 77.2930 70.6126 65.4576 61.1510 51.9531 59.2919 37.2301 59.3374 52.6177 65.6444 0.3946 0.6131       92.3312
Fine-Tuned NLLB-600M      sylheti        917 51.2084 70.3853 62.3510 56.2156 51.2084 45.1987 52.8594 29.9446 52.7923 45.5596 57.9448 0.4684 0.6901       90.5425
Fine-Tuned NLLB-600M      overall       1704 55.9127 73.6254 66.2373 60.5758 55.9127 48.3099 55.8204 33.3070 55.8022 48.8194 61.5813 0.4338 0.6551       91.3686


In [ ]:
#  Bilingual Lexical Gloss Construction
# Resolves OOV Paradox and Echo-Chamber effect 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

import re, pandas as pd
from collections import Counter, defaultdict
from typing import Dict, List
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'


def tokenize_bangla(text: str) -> List[str]:
    """Lightweight Bangla tokenizer: extract Unicode Bangla word sequences."""
    return re.findall(r'[\u0980-\u09FF]+', str(text))


def build_bilingual_gloss(
    src_texts: List[str],
    tgt_texts: List[str],
    dialect_labels: List[str],
    min_count: int = 2,
    top_k: int = 2000,
) -> Dict[str, Dict[str, str]]:
    """
    IBM Model-1 style bilingual lexical gloss:
    dialect_word → most probable standard_bangla_word.
    """
    gloss = {'chittagonian': {}, 'sylheti': {}}
    for _d in ['chittagonian', 'sylheti']:
        _idx  = [i for i, d in enumerate(dialect_labels) if d == _d]
        _cooc : Dict[str, Counter] = defaultdict(Counter)
        _freq : Counter = Counter()
        for i in _idx:
            _sw = tokenize_bangla(src_texts[i])
            _tw = tokenize_bangla(tgt_texts[i])
            _freq.update(_sw)
            for sw in _sw:
                for tw in _tw:
                    if sw != tw:  # Only dialectal shifts
                        _cooc[sw][tw] += 1
        _entries = []
        for sw, tw_counts in _cooc.items():
            if _freq[sw] < min_count: continue
            best_tw, best_cnt = tw_counts.most_common(1)[0]
            _entries.append((best_cnt / _freq[sw], sw, best_tw))
        _entries.sort(reverse=True)
        for _, sw, tw in _entries[:top_k]:
            gloss[_d][sw] = tw
    return gloss


print('[GLOSS] Building bilingual lexical gloss...')
BILINGUAL_GLOSS = build_bilingual_gloss(
    _train_sources, _train_targets, _train_dialects,
    min_count=2, top_k=2000,
)

_gloss_rows = [
    {'dialect': d, 'dialect_word': dw, 'standard_word': sw}
    for d, mapping in BILINGUAL_GLOSS.items()
    for dw, sw in mapping.items()
]
pd.DataFrame(_gloss_rows).to_csv(WORK_DIR / 'bilingual_gloss.csv', index=False)

print(f'[GLOSS] Chittagonian: {len(BILINGUAL_GLOSS["chittagonian"]):,} | Sylheti: {len(BILINGUAL_GLOSS["sylheti"]):,}')
print('[GLOSS] Sample entries:')
for dw, sw in list(BILINGUAL_GLOSS['chittagonian'].items())[:5]:
    print(f'  chitt: {dw} → std: {sw}')

[GLOSS] Building bilingual lexical gloss...
[GLOSS] Chittagonian: 2,000 | Sylheti: 2,000
[GLOSS] Sample entries:
  chitt: জনসাধারণ → std: টাকায়
  chitt: য় → std: টাকায়
  chitt: ফুত → std: পুত্র
  chitt: অর্থে → std: আলাদা
  chitt: ৫০ → std: টাকায়


In [10]:
# Full Retrieval Engine: BGE-M3 + BM25Okapi + Reranker + RRF

import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['sentence_transformers','transformers']: logging.getLogger(_n).setLevel(logging.ERROR) 

import gc, torch, numpy as np, re, pandas as pd
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

WORK_DIR = Path('/kaggle/working')
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'

#  Bilingual Gloss Helper 
def tokenize_bangla(text: str) -> List[str]:
    """Lightweight Bangla tokenizer: extract Unicode Bangla word sequences."""
    return re.findall(r'[\u0980-\u09FF]+', str(text))

def build_bilingual_gloss(
    src_texts: List[str],
    tgt_texts: List[str],
    dialect_labels: List[str],
    min_count: int = 2,
    top_k: int = 2000,
) -> Dict[str, Dict[str, str]]:
    """
    IBM Model-1 style bilingual lexical gloss:
    dialect_word → most probable standard_bangla_word.
    """
    gloss = {'chittagonian': {}, 'sylheti': {}}
    for _d in ['chittagonian', 'sylheti']:
        _idx  = [i for i, d in enumerate(dialect_labels) if d == _d]
        _cooc : Dict[str, Counter] = defaultdict(Counter)
        _freq : Counter = Counter()
        for i in _idx:
            _sw = tokenize_bangla(src_texts[i])
            _tw = tokenize_bangla(tgt_texts[i])
            _freq.update(_sw)
            for sw in _sw:
                for tw in _tw:
                    if sw != tw:  # Only dialectal shifts
                        _cooc[sw][tw] += 1
        _entries = []
        for sw, tw_counts in _cooc.items():
            if _freq[sw] < min_count: continue
            best_tw, best_cnt = tw_counts.most_common(1)[0]
            _entries.append((best_cnt / _freq[sw], sw, best_tw))
        _entries.sort(reverse=True)
        for _, sw, tw in _entries[:top_k]:
            gloss[_d][sw] = tw
    return gloss

print('[GLOSS] Building bilingual lexical gloss...')
BILINGUAL_GLOSS = build_bilingual_gloss(
    _train_sources, _train_targets, _train_dialects,
    min_count=2, top_k=2000,
)

_gloss_rows = [
    {'dialect': d, 'dialect_word': dw, 'standard_word': sw}
    for d, mapping in BILINGUAL_GLOSS.items()
    for dw, sw in mapping.items()
]
pd.DataFrame(_gloss_rows).to_csv(WORK_DIR / 'bilingual_gloss.csv', index=False)
print(f'[GLOSS] Chittagonian: {len(BILINGUAL_GLOSS["chittagonian"]):,} | Sylheti: {len(BILINGUAL_GLOSS["sylheti"]):,}')

# Retrieval Engine Definition
DEVICE_STR = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'

print('[RETRIEVER] Loading BAAI/bge-m3 dense encoder...')
_bge_m3 = SentenceTransformer('BAAI/bge-m3', device=DEVICE_STR)

print('[RETRIEVER] Loading BAAI/bge-reranker-v2-m3...')
_bge_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE_STR)


class DialectRetrievalEngine:
    def __init__(self, srcs, tgts, dialects, gloss, encoder, reranker, rrf_k=60):
        self.sources, self.targets, self.dialects = srcs, tgts, dialects
        self.gloss, self.encoder = gloss, encoder
        #  Define both attributes so retrieve() can never miss it
        self.reranker = self._reranker = reranker
        self.rrf_k = rrf_k

        print('[RETRIEVER] Building BM25 indices per dialect...')
        self._bm25 = {}
        self._bm25_idx = {}
        for _d in ['chittagonian', 'sylheti']:
            _idx = [i for i, d in enumerate(dialects) if d == _d]
            _toks = [tokenize_bangla(srcs[i]) for i in _idx]
            self._bm25[_d]     = BM25Okapi(_toks)
            self._bm25_idx[_d] = _idx

        print('[RETRIEVER] Encoding corpus with BGE-M3...')
        self._embs = encoder.encode(
            srcs, batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        print(f'[RETRIEVER] Corpus embeddings shape: {self._embs.shape}')

    def _rare_density(self, text: str, dialect: str) -> float:
        _w = tokenize_bangla(text)
        if not _w: return 0.0
        return sum(1 for w in _w if w in self.gloss.get(dialect, {})) / len(_w)

    def _select_k(self, density: float) -> int:
        """Critique 4: 0ms overhead for easy samples."""
        if density < 0.15: return 0
        if density < 0.35: return 2
        return 4

    def _dense(self, q: str, n: int) -> List[Tuple[int,float]]:
        _qe = self.encoder.encode([q], normalize_embeddings=True)[0]
        _sc = self._embs @ _qe
        _top = np.argsort(_sc)[::-1][:n]
        return [(int(i), float(_sc[i])) for i in _top]

    def _bm25_ret(self, q: str, dialect: str, n: int) -> List[Tuple[int,float]]:
        _t  = tokenize_bangla(q)
        _sc = self._bm25[dialect].get_scores(_t)
        _top = np.argsort(_sc)[::-1][:n]
        _idx_map = self._bm25_idx[dialect]
        return [(int(_idx_map[j]), float(_sc[j])) for j in _top]

    def _rrf(self, lists, k=60) -> List[int]:
        _scores: Dict[int, float] = defaultdict(float)
        for lst in lists:
            for rank, (idx, _) in enumerate(lst):
                _scores[idx] += 1.0 / (k + rank + 1)
        return sorted(_scores, key=_scores.get, reverse=True)

    def retrieve(self, query: str, dialect: str,
                 force_k: Optional[int] = None) -> Tuple[str, int, float]:
        _density = self._rare_density(query, dialect)
        _k       = force_k if force_k is not None else self._select_k(_density)

        if _k == 0:
            return '', 0, _density

        # Candidate retrieval (Dense + BM25)
        _d_res  = self._dense(query, _k * 5)
        _b_res  = self._bm25_ret(query, dialect, _k * 5)
        _fused  = self._rrf([_d_res, _b_res], k=self.rrf_k)[:_k*3]

        # Cross-encoder reranking
        _pairs    = [(query, self.sources[i]) for i in _fused]
        _rr_sc    = self._reranker.predict(_pairs, show_progress_bar=False)
        _reranked = sorted(zip(_fused, _rr_sc), key=lambda x: x[1], reverse=True)
        _top_k    = [i for i, _ in _reranked[:_k]]

        # Compact context format
        _ctx_parts = [
            f'<ctx> [SRC] {self.sources[i]} [TGT] {self.targets[i]} </ctx>'
            for i in _top_k
        ]

        # Gloss injection
        _words      = tokenize_bangla(query)
        _gloss_hits = [(w, self.gloss[dialect][w])
                       for w in _words if w in self.gloss.get(dialect, {})]
        _gloss_str  = ''
        if _gloss_hits:
            _gpairs    = '; '.join(f'{dw}→{sw}' for dw, sw in _gloss_hits)
            _gloss_str = f'<gloss> {_gpairs} </gloss> '

        _full = _gloss_str + ' '.join(_ctx_parts) + f' [QUERY] {query}'
        return _full.strip(), _k, _density


print('[RETRIEVER] Initializing DialectRetrievalEngine...')
RETRIEVAL_ENGINE = DialectRetrievalEngine(
    srcs=_train_sources, tgts=_train_targets, dialects=_train_dialects,
    gloss=BILINGUAL_GLOSS, encoder=_bge_m3, reranker=_bge_reranker, rrf_k=60,
)

_demo_ctx, _demo_k, _demo_rarity = RETRIEVAL_ENGINE.retrieve(
    _test_sources[0], _test_dialects[0]
)
print(f'\n[RETRIEVER] Demo successful ✓')
print(f'  • rare_density : {_demo_rarity:.2f}')
print(f'  • k_selected   : {_demo_k}')
print(f'  • context sample: {_demo_ctx[:150]}...')

[GLOSS] Building bilingual lexical gloss...
[GLOSS] Chittagonian: 2,000 | Sylheti: 2,000
[RETRIEVER] Loading BAAI/bge-m3 dense encoder...
[RETRIEVER] Loading BAAI/bge-reranker-v2-m3...
[RETRIEVER] Initializing DialectRetrievalEngine...
[RETRIEVER] Building BM25 indices per dialect...
[RETRIEVER] Encoding corpus with BGE-M3...
[RETRIEVER] Corpus embeddings shape: (13632, 1024)

[RETRIEVER] Demo successful ✓
  • rare_density : 0.20
  • k_selected   : 2
  • context sample: <gloss> কীতা→কী </gloss> <ctx> [SRC] আফনে আত দেখতে জানইন? [TGT] আপনি হাত দেখতে জানেন? </ctx> <ctx> [SRC] আফনে আইতা নায়? [TGT] অপনি আসবেন না? </ctx> [Q...


In [ ]:
# Adaptive RAG + Competitive Verifier on FT-NLLB 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

import gc, torch, numpy as np, pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

#  Configure GPU Device 
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"[DEVICE] Target hardware set to: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}) ✓")

WORK_DIR = Path('/kaggle/working')
FT_MODEL_DIR = WORK_DIR / 'fine_tuned_nllb600m'

def _flush_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

_flush_gpu()

#  Use In-Memory Test Set & Load Standalone Predictions
if 'test_df' in globals() and '_test_sources' in globals():
    eval_test_df = test_df.copy()
    sources  = _test_sources
    targets  = _test_targets
    dialects = _test_dialects
    print(f"[LOAD] Using pre-defined in-memory test set ({len(sources):,} samples) ✓")
elif (WORK_DIR / 'split_test.csv').exists():
    eval_test_df = pd.read_csv(WORK_DIR / 'split_test.csv')
    COL_SRC = 'source' if 'source' in eval_test_df.columns else eval_test_df.columns[0]
    COL_TGT = 'target' if 'target' in eval_test_df.columns else eval_test_df.columns[1]
    COL_DIALECT = 'dialect_type' if 'dialect_type' in eval_test_df.columns else 'dialect'
    sources  = eval_test_df[COL_SRC].astype(str).tolist()
    targets  = eval_test_df[COL_TGT].astype(str).tolist()
    dialects = eval_test_df[COL_DIALECT].astype(str).tolist()
    print(f"[LOAD] Loaded test set from {WORK_DIR / 'split_test.csv'} ({len(sources):,} samples) ✓")
else:
    raise FileNotFoundError("Could not find test_df in memory or split_test.csv on disk!")

# Load the standalone predictions as the guaranteed anchor
ft_pred_file = WORK_DIR / 'nllb_fine_tuned_predictions.csv'
if ft_pred_file.exists():
    ft_preds_df = pd.read_csv(ft_pred_file)
    pred_col = 'pred_nllb_ft' if 'pred_nllb_ft' in ft_preds_df.columns else ('nllb_fine_tuned' if 'nllb_fine_tuned' in ft_preds_df.columns else ft_preds_df.columns[-1])
    ft_fallback_preds = ft_preds_df[pred_col].astype(str).tolist()
    print(f"[LOAD] Loaded {len(ft_fallback_preds)} Fine-Tuned baseline predictions")
else:
    raise FileNotFoundError("nllb_fine_tuned_predictions.csv missing in /kaggle/working/!")

# Construct Adaptive Sequences (Instant Sub-second Pass) 
print("\n[RAG Engine] Dynamically routing test set (Dynamic k ∈ {0, 1})...")
lean_rag_inputs = []
adaptive_k_list = []

for s, d in zip(sources, dialects):
    try:
        # Check if high-speed method exists, else fallback to standard retrieve
        if hasattr(RETRIEVAL_ENGINE, 'retrieve_adaptive_exemplar'):
            top_tgt, k_val = RETRIEVAL_ENGINE.retrieve_adaptive_exemplar(s, d)
        else:
            _, k_val, _ = RETRIEVAL_ENGINE.retrieve(s, d, force_k=None)
            if k_val > 0:
                d_hits = RETRIEVAL_ENGINE._dense(s, 1)
                top_tgt = RETRIEVAL_ENGINE.targets[d_hits[0][0]]
            else:
                top_tgt = ""
        
        if k_val > 0 and len(str(top_tgt).strip()) > 0:
            # Query-first syntax with clean native Bengali boundary
            lean_rag_inputs.append(f"{s} । প্রসঙ্গ: {top_tgt}".strip())
            adaptive_k_list.append(1)
        else:
            lean_rag_inputs.append(s)
            adaptive_k_list.append(0)
    except Exception:
        lean_rag_inputs.append(s)
        adaptive_k_list.append(0)

n_ret = adaptive_k_list.count(1)
n_par = adaptive_k_list.count(0)
print(f"  • {n_ret}/{len(lean_rag_inputs)} samples assigned retrieval guidance (k=1)")
print(f"  • {n_par}/{len(lean_rag_inputs)} samples assigned pure parametric routing (k=0)")

# Load Fine-Tuned Model 
print(f"\n[MODEL] Loading Fine-Tuned NLLB-600M ")
BASE_MODEL_ID = 'facebook/nllb-200-distilled-600M'

tokenizer = AutoTokenizer.from_pretrained(str(FT_MODEL_DIR), src_lang='ben_Beng', tgt_lang='ben_Beng')
_bos_id = tokenizer.convert_tokens_to_ids('ben_Beng')

# FP16 inference 
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to(DEVICE)

ft_model = PeftModel.from_pretrained(base_model, str(FT_MODEL_DIR)).to(DEVICE)
ft_model.eval()

def _gpu_batch_generate(model, tokenizer, texts, batch_size=32):
    all_out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=160).to(DEVICE)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=128, 
                num_beams=4, 
                early_stopping=True,
                forced_bos_token_id=_bos_id
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = []
        for pred in decoded:
            txt = pred.strip()
            if "প্রসঙ্গ:" in txt:
                txt = txt.split("প্রসঙ্গ:")[0].strip()
            cleaned.append(txt)
        all_out.extend(cleaned)
    return all_out

print("[INFERENCE] Generating predictions")
raw_rag_candidates = _gpu_batch_generate(ft_model, tokenizer, lean_rag_inputs, batch_size=32)

del ft_model, base_model
_flush_gpu()

# Competitive Relative Verifier
print("\n[VERIFIER] Running Competitive Relative Verification...")

class CompetitiveVerifier:
    def __init__(self, encoder, margin=0.005):
        self.encoder = encoder
        self.margin = margin

    def verify(self, sources_list, rag_cands, ft_cands):
        # Fast batch encoding with BGE-M3
        src_embs = self.encoder.encode(sources_list, normalize_embeddings=True, show_progress_bar=False, batch_size=64)
        rag_embs = self.encoder.encode(rag_cands, normalize_embeddings=True, show_progress_bar=False, batch_size=64)
        ft_embs  = self.encoder.encode(ft_cands, normalize_embeddings=True, show_progress_bar=False, batch_size=64)

        sim_rag = np.sum(src_embs * rag_embs, axis=1)
        sim_ft  = np.sum(src_embs * ft_embs, axis=1)

        final_verified = []
        rag_accepted_count = 0

        for i in range(len(sources_list)):
            # Accept RAG only if it strictly outperforms standalone semantic fidelity
            if (sim_rag[i] > sim_ft[i] + self.margin) and (len(rag_cands[i]) >= 0.5 * len(ft_cands[i])):
                final_verified.append(rag_cands[i])
                rag_accepted_count += 1
            else:
                final_verified.append(ft_cands[i])

        print(f"  • Verifier Decision: Accepted RAG on {rag_accepted_count}/{len(sources_list)} samples.")
        print(f"  • Standalone Fallback retained on {len(sources_list) - rag_accepted_count}/{len(sources_list)} samples.")
        return final_verified

comp_verifier = CompetitiveVerifier(_bge_m3, margin=0.005)
verified_proposed_preds = comp_verifier.verify(sources, raw_rag_candidates, ft_fallback_preds)

# Evaluate Metrics Across All Dialects
print("\n[EVALUATION] Computing Unified Metrics for the Proposed System...")

proposed_metrics = evaluate_by_dialect(
    hypotheses=verified_proposed_preds,
    references=targets,
    dialect_labels=dialects,
    system_name='FT-NLLB_Adaptive-RAG+Verifier (Proposed)',
    compute_bertscore=True
)

# Update the ablation file cleanly
ablation_file = WORK_DIR / 'proposed_system_ablation_metrics.csv'
if ablation_file.exists():
    existing_ablation = pd.read_csv(ablation_file)
    existing_ablation = existing_ablation[~existing_ablation['system'].str.contains('Proposed')]
    updated_ablation = pd.concat([existing_ablation, proposed_metrics], ignore_index=True)
else:
    updated_ablation = proposed_metrics

updated_ablation.to_csv(ablation_file, index=False, encoding='utf-8-sig')
print(f"[SAVED] Updated ablation metrics saved to: {ablation_file} ✓")

# Save predictions for qualitative case studies
eval_test_df['ft_nllb_standalone'] = ft_fallback_preds
eval_test_df['proposed_verified_pred'] = verified_proposed_preds
eval_test_df['k_used'] = adaptive_k_list
eval_test_df.to_csv(WORK_DIR / 'proposed_system_predictions.csv', index=False, encoding='utf-8-sig')

# Display Updated Results 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

for d in ['overall', 'chittagonian', 'sylheti']:
    sub = updated_ablation[updated_ablation['dialect'] == d].sort_values('bleu4', ascending=False)
    print("\n" + "="*120)
    print(f"        [UPDATED PROPOSED SYSTEM ABLATION: {d.upper()}]")
    print("="*120)
    print(sub[['system', 'bleu4', 'bleu1', 'rouge1', 'rougeL', 'meteor', 'chrfpp', 'bertscore_f1']].to_string(index=False))

[DEVICE] Target hardware set to: cuda:0 (Tesla T4) ✓
[LOAD] Using pre-defined in-memory test set (1,704 samples) ✓
[LOAD] Loaded 1704 Fine-Tuned baseline predictions

[RAG Engine] Dynamically routing test set (Dynamic k ∈ {0, 1})...
  • 857/1704 samples assigned retrieval guidance (k=1)
  • 847/1704 samples assigned pure parametric routing (k=0)

[MODEL] Loading Fine-Tuned NLLB-600M 
[INFERENCE] Generating predictions

[VERIFIER] Running Competitive Relative Verification...
  • Verifier Decision: Accepted RAG on 29/1704 samples.
  • Standalone Fallback retained on 1675/1704 samples.

[EVALUATION] Computing Unified Metrics for the Proposed System...
[SAVED] Updated ablation metrics saved to: /kaggle/working/proposed_system_ablation_metrics.csv ✓

        [UPDATED PROPOSED SYSTEM ABLATION: OVERALL]
                                  system  bleu4   bleu1  rouge1  rougeL  meteor  chrfpp  bertscore_f1
FT-NLLB_Adaptive-RAG+Verifier (Proposed) 55.457 73.1181  55.584 55.5688 48.7542  61.462   

In [14]:
# Qualitative Case Studies (Saved as .xlsx & .csv)
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

# Ensure Excel writer library is present
!pip install -q openpyxl

import pandas as pd, sacrebleu, numpy as np
from pathlib import Path

WORK_DIR = Path('/kaggle/working')

def _s_bleu4(h: str, r: str) -> float:
    try:
        return sacrebleu.corpus_bleu([str(h).strip()], [[str(r).strip()]], smooth_method='exp', tokenize='flores200').score
    except Exception:
        return 0.0

# Load Predictions 
# Uses ft_rag_preds_df from the previous cell, or loads from disk
if 'ft_rag_preds_df' in globals():
    preds_df = ft_rag_preds_df.copy()
elif (WORK_DIR / 'proposed_system_predictions.csv').exists():
    preds_df = pd.read_csv(WORK_DIR / 'proposed_system_predictions.csv')
else:
    raise FileNotFoundError("Could not find proposed_system_predictions.csv in /kaggle/working/")

# Stratified Sampling Across Difficulty (12 Total Cases)
def _sample_strata(df, dialect, n_easy=2, n_med=2, n_hard=2):
    sub = df[df['dialect_type'] == dialect].copy()
    
    # Calculate word length for difficulty tiering
    sub['src_len'] = sub['source'].astype(str).str.split().str.len()
    sub['rare_k']  = sub['k_used'] if 'k_used' in sub.columns else 2
    
    # Define difficulty tiers
    sub['difficulty'] = 'medium'
    sub.loc[(sub['rare_k'] == 0) & (sub['src_len'] <= 6), 'difficulty'] = 'easy'
    sub.loc[(sub['rare_k'] >= 4) | (sub['src_len'] >= 12), 'difficulty'] = 'hard'
    
    samples = []
    for tier, n in [('easy', n_easy), ('medium', n_med), ('hard', n_hard)]:
        pool = sub[sub['difficulty'] == tier]
        if len(pool) > 0:
            samples.append(pool.sample(n=min(n, len(pool)), random_state=42))
        else:
            samples.append(sub.sample(n=min(n, len(sub)), random_state=42))
    return pd.concat(samples, ignore_index=True)

case_rows = []
for dialect in ['chittagonian', 'sylheti']:
    sampled = _sample_strata(preds_df, dialect)
    for idx, (_, row) in enumerate(sampled.iterrows(), 1):
        src = str(row['source'])
        ref = str(row['target'])
        
        ft_pred   = str(row.get('ft_nllb_standalone', ''))
        rag_pred  = str(row.get('ft_nllb_adaptive_verif', ''))
        bm25_pred = str(row.get('ft_nllb_bm25_rag', ''))
        
        case_rows.append({
            'Dialect'                   : dialect.capitalize(),
            'Case_ID'                   : f"{dialect[:4].upper()}_{idx:02d}",
            'Difficulty'                : row['difficulty'].upper(),
            'Dialect_Source'            : src,
            'Ground_Truth_Standard'     : ref,
            'FineTuned_NLLB_Alone'      : ft_pred,
            'Proposed_Adaptive_RAG'     : rag_pred,
            'BLEU4_FineTuned'           : round(_s_bleu4(ft_pred, ref), 2),
            'BLEU4_Proposed_RAG'        : round(_s_bleu4(rag_pred, ref), 2),
            'k_Retrieved'               : row.get('k_used', 0)
        })

case_study_df = pd.DataFrame(case_rows)

# Save as both .xlsx (Excel Native) and .csv (BOM-encoded) ──
case_study_df.to_excel(WORK_DIR / 'qualitative_case_studies.xlsx', index=False, engine='openpyxl')
case_study_df.to_csv(WORK_DIR / 'qualitative_case_studies.csv', index=False, encoding='utf-8-sig')

print(f"[CASE-STUDY] Saved Native Excel : {WORK_DIR / 'qualitative_case_studies.xlsx'} ✓")
print(f"[CASE-STUDY] Saved UTF-8 CSV    : {WORK_DIR / 'qualitative_case_studies.csv'} ✓")

# Display Formatted Translation Samples 
print("\n" + "="*120)
print("                                [QUALITATIVE TRANSLATION CASE STUDIES]")
print("="*120)
for _, r in case_study_df.iterrows():
    print(f"\n[{r['Case_ID']}] Dialect: {r['Dialect']} | Tier: {r['Difficulty']} | k={r['k_Retrieved']}")
    print(f"  • Source (Dialect)    : {r['Dialect_Source']}")
    print(f"  • Target (Ground Truth): {r['Ground_Truth_Standard']}")
    print(f"  • Fine-Tuned NLLB     : {r['FineTuned_NLLB_Alone']}  (BLEU-4: {r['BLEU4_FineTuned']})")
    print(f"  • Proposed Full RAG   : {r['Proposed_Adaptive_RAG']}  (BLEU-4: {r['BLEU4_Proposed_RAG']})")

[CASE-STUDY] Saved Native Excel : /kaggle/working/qualitative_case_studies.xlsx ✓
[CASE-STUDY] Saved UTF-8 CSV    : /kaggle/working/qualitative_case_studies.csv ✓

                                [QUALITATIVE TRANSLATION CASE STUDIES]

[CHIT_01] Dialect: Chittagonian | Tier: EASY | k=0
  • Source (Dialect)    : ঠেং নাচানি থাইমমি ।
  • Target (Ground Truth): পা নাচানো থামল।
  • Fine-Tuned NLLB     : ঠেং নাচানো থামব।  (BLEU-4: 41.11)
  • Proposed Full RAG   :   (BLEU-4: 0.0)

[CHIT_02] Dialect: Chittagonian | Tier: EASY | k=0
  • Source (Dialect)    : মাইয়া নাই, তিন পোয়া।
  • Target (Ground Truth): মেয়ে নাই, তিন ছেলে।
  • Fine-Tuned NLLB     : মেয়ে নেই, তিন পায়ে।  (BLEU-4: 15.62)
  • Proposed Full RAG   :   (BLEU-4: 0.0)

[CHIT_03] Dialect: Chittagonian | Tier: MEDIUM | k=1
  • Source (Dialect)    : কোটা আন্দোলনর শুরু ত্থুন্ ই ইতা সক্রিয় আছিল।
  • Target (Ground Truth): কোটা আন্দোলনের শুরু থেকেই সে সক্রিয় ছিল।
  • Fine-Tuned NLLB     : কোটা আন্দোলনের শুরু থেকেই তিনি সক্রিয় ছিলেন।  (BL